# Week 57, Day 1: Review: structures

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w057d1.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
from collections import deque
import heapq

d = deque([1, 2, 3], maxlen=3)
d.append(4)
h = [5, 3, 8, 1]
heapq.heapify(h)
s = {3, 1, 2}
print(list(d), h[0], 2 in s, sorted(s))

In [ ]:
def longest_zero_sum(nums):
    first = {0: -1}                       # prefix total -> first position it was seen
    total, best = 0, 0
    for i, x in enumerate(nums):
        total += x
        if total in first:
            best = max(best, i - first[total])
        else:
            first[total] = i
    return best

print(longest_zero_sum([1, -1, 3, 2, -2, -3, 3]), longest_zero_sum([1, 2, 3]))

In [ ]:
class Node:
    def __init__(self, value, next=None):
        self.value, self.next = value, next

def build(values):
    head = None
    for v in reversed(values):
        head = Node(v, head)
    return head

def to_list(head):
    out = []
    while head:
        out.append(head.value)
        head = head.next
    return out

def odd_then_even(head):
    if not head:
        return head
    odd, even = head, head.next
    even_head = even
    while even and even.next:
        odd.next = even.next
        odd = odd.next
        even.next = odd.next
        even = even.next
    odd.next = even_head
    return head

print(to_list(odd_then_even(build([1, 2, 3, 4, 5]))), to_list(odd_then_even(build([7]))))

In [ ]:
import heapq

def run_order(jobs):
    """jobs: (arrival, length). Shortest waiting job first; ties by index."""
    order = sorted(range(len(jobs)), key=lambda i: jobs[i][0])
    waiting, out, time, k = [], [], 0, 0
    while len(out) < len(jobs):
        while k < len(order) and jobs[order[k]][0] <= time:
            i = order[k]
            heapq.heappush(waiting, (jobs[i][1], i))
            k += 1
        if not waiting:
            time = jobs[order[k]][0]           # the machine is idle until the next arrival
            continue
        length, i = heapq.heappop(waiting)
        out.append(i)
        time += length
    return out

print(run_order([(1, 2), (2, 4), (3, 2), (4, 1)]))

> **Key idea.** Choose a structure from the operations a problem needs and how often. Lists index and append in O(1) but shift on inserts; deques are O(1) at both ends; dicts and sets look up in O(1) on average; balanced BSTs keep order with O(log n) operations; heaps give the smallest in O(1) and pop it in O(log n); tries make prefixes cheap; union-find merges groups in nearly O(1). Combine structures when operations pull in different directions. Prefix sums with a dict of first positions find zero-sum stretches; circular buffers keep the last n items; walk tall trees with an explicit stack.

### Exercise: A circular buffer

Write a class `Ring(capacity)` keeping the last `capacity` items in a fixed-size list (no deque, no slicing to remove old items): `add(x)` overwrites the oldest when full; `len(r)`; `r[i]` gives the i-th oldest item (negative i from the newest; `IndexError` out of range); `items()` returns them oldest first.

In [ ]:
class Ring:
    def __init__(self, capacity):
        ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\nfrom collections import deque\ndef test_ring():\n    \"circular buffer\"\n    r = Ring(3)\n    for x in range(5):\n        r.add(x)\n    assert r.items() == [2, 3, 4] and len(r) == 3 and r[0] == 2 and r[-1] == 4\n    try:\n        r[3]\n        assert False\n    except IndexError:\n        pass\n    rng = random.Random(1)\n    for _ in range(50):\n        cap = rng.randint(1, 5)\n        r, ref = Ring(cap), deque(maxlen=cap)\n        for _ in range(rng.randint(0, 20)):\n            x = rng.randint(0, 99)\n            r.add(x)\n            ref.append(x)\n            assert r.items() == list(ref) and len(r) == len(ref)\n            i = rng.randrange(-len(ref), len(ref))\n            assert r[i] == ref[i]\n    assert \"deque\" not in source")

### Exercise: Balanced stretches

Write `longest_zero_sum(nums)`: the length of the longest stretch of neighbouring numbers adding up to 0, in one pass with a dict of first positions. Then `longest_balanced(text)`: the longest stretch of a string of `"H"` and `"T"` (coin tosses) with as many heads as tails (count H as +1 and T as −1).

In [ ]:
def longest_zero_sum(nums):
    ___

def longest_balanced(text):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef test_zero():\n    \"prefix sums and a dict\"\n    assert longest_zero_sum([1, -1, 3, 2, -2, -3, 3]) == 6 and longest_zero_sum([1, 2]) == 0 and longest_zero_sum([]) == 0\n    assert longest_balanced(\"HHTHTT\") == 6 and longest_balanced(\"HHH\") == 0 and longest_balanced(\"THHT\") == 4\n    rng = random.Random(2)\n    for _ in range(200):\n        nums = [rng.randint(-3, 3) for _ in range(rng.randint(0, 10))]\n        assert longest_zero_sum(nums) == max([j - i for i in range(len(nums)) for j in range(i + 1, len(nums) + 1) if sum(nums[i:j]) == 0], default=0)\n        s = \"\".join(rng.choice(\"HT\") for _ in range(rng.randint(0, 10)))\n        assert longest_balanced(s) == max([j - i for i in range(len(s)) for j in range(i + 1, len(s) + 1) if s[i:j].count(\"H\") * 2 == j - i], default=0)\n    assert longest_zero_sum([1, -1] * 100_000) == 200_000")

### Exercise: Odd places first

Using the `Node` class given, write `odd_then_even(head)`: regroup a linked list so the nodes in positions 1, 3, 5, ... come first, then 2, 4, 6, ..., each group in its original order, in one pass and without creating nodes. Then `middle(head)`: the value of the middle node (the second middle for an even length), with fast and slow pointers.

In [ ]:
class Node:
    def __init__(self, value, next=None):
        self.value, self.next = value, next

def odd_then_even(head):
    ___

def middle(head):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef build(values):\n    head = None\n    for v in reversed(values):\n        head = Node(v, head)\n    return head\ndef to_list(head):\n    out = []\n    while head:\n        out.append(head.value)\n        head = head.next\n    return out\ndef test_links():\n    \"splicing\"\n    assert to_list(odd_then_even(build([1, 2, 3, 4, 5]))) == [1, 3, 5, 2, 4] and odd_then_even(None) is None\n    assert middle(build([1, 2, 3, 4])) == 3 and middle(build([9])) == 9\n    rng = random.Random(3)\n    for _ in range(100):\n        vals = [rng.randint(0, 9) for _ in range(rng.randint(1, 12))]\n        head = build(vals)\n        nodes = []\n        n = head\n        while n:\n            nodes.append(n)\n            n = n.next\n        out = odd_then_even(head)\n        assert to_list(out) == vals[0::2] + vals[1::2]\n        got = []\n        while out:\n            got.append(out)\n            out = out.next\n        assert {id(x) for x in got} == {id(x) for x in nodes}\n        assert middle(build(vals)) == vals[len(vals) // 2]")

### Exercise: Rebuilding a search tree

A BST's keys were written down in **pre-order** (node, left subtree, right subtree). Write `from_preorder(keys)`: rebuild the tree as nested `[key, left, right]` lists (`None` for empty), in O(n), with an explicit stack: each new key is the left child of the stack's top if smaller, otherwise the right child of the last node popped while keys were smaller. Then `kth_smallest(tree, k)` (k from 1) with an iterative in-order walk.

In [ ]:
def from_preorder(keys):
    ___

def kth_smallest(tree, k):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef insert(t, k):\n    if t is None:\n        return [k, None, None]\n    if k < t[0]:\n        t[1] = insert(t[1], k)\n    else:\n        t[2] = insert(t[2], k)\n    return t\ndef pre(t, out):\n    if t:\n        out.append(t[0])\n        pre(t[1], out)\n        pre(t[2], out)\n    return out\ndef test_bst():\n    \"pre-order and in-order\"\n    t = from_preorder([8, 5, 1, 7, 10, 12])\n    assert t == [8, [5, [1, None, None], [7, None, None]], [10, None, [12, None, None]]]\n    assert from_preorder([]) is None and kth_smallest(t, 1) == 1 and kth_smallest(t, 6) == 12\n    rng = random.Random(4)\n    for _ in range(100):\n        keys = rng.sample(range(100), rng.randint(1, 20))\n        tree = None\n        for k in keys:\n            tree = insert(tree, k)\n        assert from_preorder(pre(tree, [])) == tree\n        k = rng.randint(1, len(keys))\n        assert kth_smallest(tree, k) == sorted(keys)[k - 1]\n    tall = from_preorder(list(range(5000, 0, -1)))\n    assert kth_smallest(tall, 4999) == 4999")

### Exercise: Shortest job next

Write `run_order(jobs)`: jobs are `(arrival, length)`; one machine runs them one at a time without interruption, and whenever it's free it starts the **shortest** job that has arrived (ties: the lower index; if nothing has arrived, it waits for the next arrival). Return the job indices in the order they run. Then `average_wait(jobs)`: the mean time from arrival to starting, rounded to 2 places.

In [ ]:
import heapq

def run_order(jobs):
    ___

def average_wait(jobs):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef test_schedule():\n    \"a heap of waiting jobs\"\n    jobs = [(1, 2), (2, 4), (3, 2), (4, 1)]\n    assert run_order(jobs) == [0, 2, 3, 1] and average_wait(jobs) == 1.25\n    assert run_order([(5, 1), (0, 3)]) == [1, 0] and run_order([]) == [] and average_wait([(0, 4)]) == 0.0\n    rng = random.Random(5)\n    for _ in range(100):\n        jobs = [(rng.randint(0, 10), rng.randint(1, 5)) for _ in range(rng.randint(1, 8))]\n        done, time, order = set(), 0, []\n        while len(order) < len(jobs):\n            ready = [i for i in range(len(jobs)) if i not in done and jobs[i][0] <= time]\n            if not ready:\n                time = min(jobs[i][0] for i in range(len(jobs)) if i not in done)\n                continue\n            i = min(ready, key=lambda i: (jobs[i][1], i))\n            order.append(i)\n            done.add(i)\n            time += jobs[i][1]\n        assert run_order(jobs) == order\n    big = [(i, (i * 7) % 13 + 1) for i in range(50_000)]\n    assert len(run_order(big)) == 50_000")

### Exercise: Words built a letter at a time

Write `longest_buildable(words)`: the longest word in the list that can be built one letter at a time, adding at the **end**, with every intermediate word also in the list ("w", "wo", "wor", "worl", "world"). On a tie, the alphabetically first. Use a trie of dicts (a key `"$"` marking ends) and walk only through nodes that end words. `""` if none.

In [ ]:
def longest_buildable(words):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef test_build():\n    \"a trie walk\"\n    assert longest_buildable([\"w\", \"wo\", \"wor\", \"worl\", \"world\"]) == \"world\"\n    assert longest_buildable([\"a\", \"banana\", \"app\", \"appl\", \"ap\", \"apply\", \"apple\"]) == \"apple\"\n    assert longest_buildable([\"b\", \"abc\"]) == \"b\" and longest_buildable([\"xy\"]) == \"\" and longest_buildable([]) == \"\"\n    rng = random.Random(6)\n    for _ in range(100):\n        words = list({\"\".join(rng.choice(\"ab\") for _ in range(rng.randint(1, 4))) for _ in range(rng.randint(0, 12))})\n        ws = set(words)\n        ok = [w for w in words if all(w[:i] in ws for i in range(1, len(w) + 1))]\n        want = min(ok, key=lambda w: (-len(w), w)) if ok else \"\"\n        assert longest_buildable(words) == want")

### Exercise: Merging contact lists

Each contact record is a name and a list of email addresses. Records sharing **any** address belong to the same person. Write `merge_contacts(records)`: a list of `(name, sorted addresses)` for each person, sorted by first address, using union-find over addresses (the name is the first record's).

In [ ]:
def merge_contacts(records):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef test_merge():\n    \"union-find\"\n    recs = [(\"Ada\", [\"a@x\", \"b@x\"]), (\"Ada\", [\"c@x\"]), (\"Ada\", [\"b@x\", \"d@x\"]), (\"Bo\", [\"e@x\"])]\n    assert merge_contacts(recs) == [(\"Ada\", [\"a@x\", \"b@x\", \"d@x\"]), (\"Ada\", [\"c@x\"]), (\"Bo\", [\"e@x\"])]\n    assert merge_contacts([]) == []\n    rng = random.Random(7)\n    for _ in range(80):\n        recs = [(\"P\", rng.sample([f\"m{i}\" for i in range(10)], rng.randint(1, 3))) for _ in range(rng.randint(1, 6))]\n        groups = [set(r[1]) for r in recs]\n        changed = True\n        while changed:\n            changed = False\n            for i in range(len(groups)):\n                for j in range(i + 1, len(groups)):\n                    if groups[i] & groups[j]:\n                        groups[i] |= groups.pop(j)\n                        changed = True\n                        break\n                if changed:\n                    break\n        want = sorted(((\"P\", sorted(g)) for g in groups), key=lambda p: p[1][0])\n        assert merge_contacts(recs) == want")

### Bug hunt: A queue that's a stack

A queue must serve the **oldest** item first. This uses `pop()` on a deque, which takes the newest, so customers are served last-come, first-served and it prints `['cai', 'ben', 'ada']`. Use `popleft()` so it prints `['ada', 'ben', 'cai']`.

In [ ]:
from collections import deque

q = deque()
for name in ("ada", "ben", "cai"):
    q.append(name)
print([q.pop() for _ in range(3)])

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"first in, first out\"\n    expect_output(\"['ada', 'ben', 'cai']\")")

### Bug hunt: The latest position, not the first

For the longest zero-sum stretch, each prefix total's **first** position must be kept. This overwrites it every time the total recurs, so the stretches it measures are too short: it prints `4` instead of `6` for `[1, -1, 3, 2, -2, -3, 3]`.

In [ ]:
def longest_zero_sum(nums):
    first, total, best = {0: -1}, 0, 0
    for i, x in enumerate(nums):
        total += x
        if total in first:
            best = max(best, i - first[total])
        first[total] = i
    return best

print(longest_zero_sum([1, -1, 3, 2, -2, -3, 3]))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"keep the first\"\n    expect_output(\"6\")")

### Bug hunt: An insert that loses the tree

Recursive BST insertion must **return** the subtree's root so the parent can relink it. This forgets the final return, so inserting 9 under 8 replaces the 8 with `None`, losing it, and this prints `[2, 5]`. Return `node` at the end so it prints `[2, 5, 8, 9]`.

In [ ]:
def insert(node, key):
    if node is None:
        return [key, None, None]
    if key < node[0]:
        node[1] = insert(node[1], key)
    else:
        node[2] = insert(node[2], key)

def in_order(node):
    return in_order(node[1]) + [node[0]] + in_order(node[2]) if node else []

root = [5, None, None]
for k in (8, 2, 9):
    insert(root, k)
print(in_order(root))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"return the subtree\"\n    expect_output(\"[2, 5, 8, 9]\")")

### Puzzle: Iterative in-order

Arrange the lines of an iterative in-order walk of a tree of `[key, left, right]` lists. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # the cost of inserting at the front of a Python list of n items: "O(1)", "O(log n)" or "O(n)"
q = ___   # the cost of popping the smallest from a heap: "O(1)", "O(log n)" or "O(n)"
r = ___   # the structure for "which group is this in?" with only merges: "trie", "union-find" or "heap"
s = ___   # list(deque([1, 2, 3], maxlen=2))
t = ___   # True or False: a dict keeps its keys sorted

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == \"O(n)\" and q == \"O(log n)\" and r == \"union-find\", \"p, q and r\"\n    assert s == [2, 3] and t is False, \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a store that remembers

A settings store keeps every value a key has ever had, with the time it was set. Write `TimeStore()`:

- `set(key, value, t)`: record the value at time t (times for a key always increase).
- `get(key, t)`: the value the key had at time t: the one set most recently at or before t; `None` if the key had no value yet. O(log n) per call, with `bisect` on each key's list of times.
- `history(key, start, end)`: the list of `(t, value)` set in the time range start to end inclusive, with two binary searches.
- `snapshot(t)`: a dict of every key's value at time t (keys with no value yet left out).

In [ ]:
import bisect

class TimeStore:
    def __init__(self):
        ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef test_store():\n    \"bisect over times\"\n    s = TimeStore()\n    s.set(\"theme\", \"light\", 1)\n    s.set(\"theme\", \"dark\", 5)\n    s.set(\"font\", 12, 3)\n    assert s.get(\"theme\", 0) is None and s.get(\"theme\", 1) == \"light\" and s.get(\"theme\", 4) == \"light\" and s.get(\"theme\", 9) == \"dark\"\n    assert s.get(\"lang\", 9) is None and s.history(\"theme\", 1, 4) == [(1, \"light\")] and s.history(\"theme\", 0, 9) == [(1, \"light\"), (5, \"dark\")]\n    assert s.snapshot(4) == {\"theme\": \"light\", \"font\": 12} and s.snapshot(0) == {}\n    rng = random.Random(8)\n    for _ in range(30):\n        st, log = TimeStore(), {}\n        t = 0\n        for _ in range(40):\n            t += rng.randint(1, 3)\n            k = rng.choice(\"abc\")\n            v = rng.randint(0, 99)\n            st.set(k, v, t)\n            log.setdefault(k, []).append((t, v))\n        for _ in range(20):\n            q = rng.randint(0, t + 2)\n            for k in \"abcd\":\n                earlier = [v for tt, v in log.get(k, []) if tt <= q]\n                assert st.get(k, q) == (earlier[-1] if earlier else None)\n            lo, hi = sorted((rng.randint(0, t), rng.randint(0, t)))\n            assert st.history(\"a\", lo, hi) == [(tt, v) for tt, v in log.get(\"a\", []) if lo <= tt <= hi]\n    big = TimeStore()\n    for i in range(100_000):\n        big.set(\"k\", i, i * 2)\n    assert big.get(\"k\", 123_457) == 61_728")

## How to review

As in every review week: predict **before** running, find exactly why when you're wrong, and keep a list of what surprised you. That list is your revision plan for Thursday's weak-spot drills.

## The cost table

| Structure | Fast | Slow | Week |
|---|---|---|---|
| list (dynamic array) | index, append, pop from the end: O(1) | insert or delete at the front or middle: O(n); search: O(n) | 41 |
| linked list | insert or delete at a known node: O(1) | index, search: O(n) | 42 |
| stack (list) | push, pop, peek: O(1) | anything in the middle | 43 |
| queue (deque) | append and pop at both ends: O(1) | index in the middle: O(n) | 43 |
| dict, set | insert, delete, lookup: O(1) on average | ordering by key: needs sorting | 44 |
| balanced BST | insert, delete, search, next larger: O(log n) | (all O(log n)) | 46 |
| heap | push, pop the smallest: O(log n); peek: O(1) | search, delete an arbitrary item: O(n) | 47 |
| trie | insert or find a word of length L: O(L); all words with a prefix | memory | 48 |
| union-find | union, find: nearly O(1) | splitting groups apart: unsupported | 51 |

The usual reasoning: list the operations the problem needs, and how often; choose the structure that makes the frequent ones fast. If two operations pull in different directions (fast lookup **and** order), combine structures: a dict plus a heap, a dict plus a linked list (an LRU cache), a sorted list plus bisect.

## Arrays and hashing

A list is a block of slots with spare room at the end, so appending is O(1) on average (Week 41). Its patterns: **two pointers**, **sliding windows**, **prefix sums**. Hashing (Week 44) turns "have I seen this?" into O(1), and with prefix sums answers "a subarray summing to k" in one pass: store each prefix total's **first** position, and a later equal total means the stretch between sums to 0.

A **circular buffer** keeps the last n items in a fixed-size list, overwriting the oldest: a `start` index and a count, with positions taken mod the capacity. `deque(maxlen=n)` does the same.

## Links, stacks and queues

Linked lists (Week 42) trade indexing for cheap splicing: keep a **dummy head**, save `node.next` before changing it, and use fast and slow pointers for middles and cycles. Stacks (Week 43) handle anything matched or undone in reverse; queues anything first come, first served; a **monotonic stack** finds the next greater element for every position in O(n).

A common splice: regroup a list so the nodes in odd positions come first, then the even ones, keeping each group's order. Two running tails, one pass, no new nodes.

## Trees and heaps

**Binary search trees** (Week 46) keep keys in order: in-order traversal lists them sorted, and search costs O(height), which balancing keeps near log n. Walk with an explicit stack when trees may be tall. **Heaps** (Week 47) give the smallest item in O(1) and remove it in O(log n): use them whenever you repeatedly need the best of a changing set (scheduling, the k largest, Dijkstra).

Scheduling with a heap: jobs arrive at given times with given lengths; one machine runs them without interruption, always starting the **shortest** waiting job next. A heap of waiting jobs, keyed by length, and a pointer through the jobs sorted by arrival.

## Tries and union-find

A **trie** (Week 48) stores words letter by letter, so every word with a prefix sits under one node: autocomplete, spell checking, and "can this word be built one letter at a time from others in the list?". **Union-find** (Week 51) keeps groups that only ever merge: `find` follows parents to a root (with path compression), `union` links two roots. Anything shaped "these belong together; are those two together?" is union-find.

## Practice

## Quiz

## Daily challenge

## Before you go

Without looking back, write the cost table from memory: each structure, its fast operations and its slow ones. Check it against today's table and add anything you missed to your revision list. Tomorrow: the algorithms.